# Phase 6A: Hybrid Feature Alignment and Training Dataset Construction

**Project**: Enhanced Legal Document Similarity and Case Retrieval Using Hybrid BERT, Legal Knowledge, and Ensemble Learning  
**Dataset**: Indian Legal Prior Case Retrieval (IL-TUR PCR)  
**Objective**: Align multi-modal pairwise retrieval features (Lexical, Dense Semantic, and Legal Knowledge) into reproducible training and evaluation datasets for downstream learning-to-rank fusion models.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Ensure project root is in path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ranking.negative_sampler import NegativeSampler
from src.ranking.feature_normalizer import FeatureNormalizer, ALL_FEATURE_COLUMNS
from src.ranking.feature_dataset_builder import FeatureDatasetBuilder

print("All Phase 6A ranking and feature alignment modules loaded successfully!")

## 1. Feature Modalities Overview

The aligned hybrid dataset combines **20 orthogonal features** spanning three distinct retrieval paradigms:
- **Lexical Retrieval**: `bm25_score`, `bm25_rank`, `tfidf_score`, `tfidf_rank`
- **Dense Semantic Retrieval**: `inlegalbert_sim`, `inlegalbert_rank` (and `bert_base_sim` on dev)
- **Legal Knowledge**: `statute_jaccard`, `statute_containment_query`, `statute_shared_count`, `citation_jaccard`, `citation_shared_count`, `entity_jaccard`, `judge_overlap`, `court_overlap`, `org_overlap`, `rhetorical_cosine`, `rhetorical_js_sim`, `rhetorical_ratio_diff`, `composite_legal_score`

In [ ]:
print(f"Total Primary Feature Columns: {len(ALL_FEATURE_COLUMNS)}")
for idx, col in enumerate(ALL_FEATURE_COLUMNS, 1):
    print(f"  {idx:2d}. {col}")

## 2. Deterministic Negative Sampling (Training Set)

Because full cross-product training contains ~3.57 million pairs with only 5,308 positives (1:672 imbalance), the `NegativeSampler` mines:
1. **100% of all valid positive pairs** (annotated precedent links in pool).
2. **Top-10 Hard Negatives** ranked highest by BM25 lexical similarity (superficial distractors).
3. **2 Random Negatives** for global non-relevance regularization.
4. **Query Self-Exclusion**: Strictly excludes $q_{\text{id}} == c_{\text{id}}$.

In [ ]:
sampler = NegativeSampler(n_hard_negatives=5, n_random_negatives=2, random_seed=42)

# Demonstration on synthetic query
q_id = "Query_101"
pos_cands = {"Cand_1", "Cand_2"}
all_cands = [f"Cand_{i}" for i in range(1, 20)]
bm25_scores = np.linspace(25.0, 1.0, len(all_cands))

sample_res = sampler.sample_candidates_for_query(q_id, pos_cands, all_cands, bm25_scores)

print(f"Query ID: {q_id}")
print(f"  Positives Retained: {sample_res['positive_count']}")
print(f"  Hard Negatives:     {sample_res['hard_negative_count']}")
print(f"  Random Negatives:   {sample_res['random_negative_count']}")
print("Sampled Candidates:")
for cid, lbl, ntype in zip(sample_res['selected_candidate_ids'], sample_res['labels'], sample_res['negative_types']):
    print(f"  • {cid:<10} | Label: {lbl} | Type: {ntype}")

## 3. Zero-Leakage Normalization Pipeline

The `FeatureNormalizer` fits transformation parameters (means, standard deviations, mins, maxs) **strictly on the training partition**, and reuses them for `dev` and `test` without estimating new statistics:
- Highly skewed counts $\rightarrow \log(1 + x)$
- Ordinal ranks $\rightarrow 1 / \text{rank}$ (reciprocal rank)
- Continuous scores $\rightarrow$ z-score standardization
- Raw feature values are permanently preserved as `raw_{col}` alongside `norm_{col}`.

In [ ]:
stats_file = PROJECT_ROOT / "results" / "tables" / "hybrid_feature_statistics.json"
if stats_file.exists():
    with open(stats_file, "r", encoding="utf-8") as f:
        stats = json.load(f)
    print("=== Dataset Sizes Across Splits ===")
    print(pd.Series(stats["dataset_sizes"]))

## 4. Feature Correlation with Target Precedent Label

Inspecting the Pearson correlation between aligned features and the ground-truth relevance label (`label = 1`).

In [ ]:
corr_file = PROJECT_ROOT / "results" / "tables" / "hybrid_feature_correlations.csv"
if corr_file.exists():
    corr_df = pd.read_csv(corr_file, index_col=0)
    label_corr = corr_df["label"].drop("label").sort_values(ascending=False)
    print("=== Top Feature Correlations with Precedent Label ===")
    display(label_corr.to_frame("Pearson Correlation with Label"))

## 5. Parquet Dataset Inspection

Loading a sample of the generated hybrid feature Parquet dataset.

In [ ]:
dev_pq = PROJECT_ROOT / "data" / "processed" / "IL-TUR-PCR" / "dev" / "hybrid_features.parquet"
if dev_pq.exists():
    dev_sample = pd.read_parquet(dev_pq)
    print(f"Dev Hybrid Dataset Shape: {dev_sample.shape}")
    print("Sample Positive Precedent Pair:")
    pos_sample = dev_sample[dev_sample["label"] == 1].head(1)
    display(pos_sample.T)